# Bible Mapping Visualisation
Explores `leftover_greek`, `leftover_latvian` and overall mapping coverage across books and chapters.

## configuration

In [ ]:
prefix_outputs="0_leftovers_out/0_LO_"

## all graphs

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BIBLE_DIR = Path('../bible')  # adjust if needed
print(f"Bible dir exists: {BIBLE_DIR.exists()}")

In [ ]:
# ── Load all chapters ──────────────────────────────────────────────────────────
records = []

for book_dir in sorted(BIBLE_DIR.iterdir()):
    if not book_dir.is_dir():
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
            print(f"  skip {json_file}: {e}")
            continue

        total_hb_words = 0
        total_has_a_greek_mapped = 0
        total_has_a_latvian_mapped = 0
        total_greek_words = 0
        total_latvian_words = 0
        leftover_greek_words = 0
        leftover_latvian_words = 0

        for verse in verses:
            for hw in verse.get('hebrew_words', []):
                total_hb_words += 1
                if hw.get('greek'):  
                    total_has_a_greek_mapped += 1
                    total_greek_words += len(hw.get('greek'))
                if hw.get('latvian'): 
                    total_has_a_latvian_mapped += 1
                    total_latvian_words += len(hw.get('latvian'))
            leftover_greek_words  += len(verse.get('leftover_greek',  []))
            total_greek_words  += len(verse.get('leftover_greek',  []))
            leftover_latvian_words += len(verse.get('leftover_latvian', []))
            total_latvian_words += len(verse.get('leftover_latvian', []))

        records.append({
            'book': book,
            'chapter': chapter,
            'total_hb_words': total_hb_words,
            'has_a_greek_mapped': total_has_a_greek_mapped,
            'has_a_latvian_mapped': total_has_a_latvian_mapped,
            'leftover_greek': leftover_greek_words,
            'total_greek_words': total_greek_words,
            'leftover_latvian': leftover_latvian_words,
            'total_latvian_words': total_latvian_words
        })

df = pd.DataFrame(records)
df['greek_usage_pct'] = (df['total_greek_words'] - df['leftover_greek'])/df['total_greek_words']*100
df['latvian_usage_pct'] = (df['total_latvian_words'] - df['leftover_latvian'])/df['total_latvian_words']*100
df['greek_pct']   = df['has_a_greek_mapped']   / df['total_hb_words'].replace(0, pd.NA) * 100
df['latvian_pct'] = df['has_a_latvian_mapped'] / df['total_hb_words'].replace(0, pd.NA) * 100
df['has_leftover_greek']   = df['leftover_greek']   > 0
df['has_leftover_latvian'] = df['leftover_latvian'] > 0
df['has_both_leftovers']   = df['has_leftover_greek'] & df['has_leftover_latvian']

print(f"Loaded {len(df)} chapters across {df['book'].nunique()} books")
df.head()

In [ ]:
t_h=df['total_hb_words'].sum()
t_g=df['total_greek_words'].sum()
t_l=df['total_latvian_words'].sum()
l_g=df['leftover_greek'].sum()
l_l=df['leftover_latvian'].sum()
a_g=df['greek_usage_pct'].mean()#.avg()
a_l=df['latvian_usage_pct'].mean()#.avg()
mn_g=df['greek_usage_pct'].min()#.avg()
mn_l=df['latvian_usage_pct'].min()#.avg()
mx_g=df['greek_usage_pct'].max()#.avg()
mx_l=df['latvian_usage_pct'].max()#.avg()
print(f" stats:\n\
    h: {t_h}\n\
    g: {t_g}\n\
    l: {t_l}\n\
    ")
print(f" leftovers:\n\
    g: {l_g}\n\
    l: {l_l}\n\
    ")
print(f" usages:\n\
    g: avg: {a_g:.2f} min: {mn_g:.2f} max: {mx_g:.2f}\n\
    l: avg: {a_l:.2f} min: {mn_l:.2f} max: {mx_l:.2f}\n\
    ")


In [ ]:
df.query(
    "greek_usage_pct==100"
)

In [ ]:
len(df.query(
    "greek_usage_pct==100"
))/len(df)

In [ ]:
len(df.query(
    "latvian_usage_pct==100"
))/len(df)

In [ ]:
len(df.query(
    "latvian_usage_pct==100 & greek_usage_pct==100"
))/len(df)

## 1. Overall mapping coverage — Pie chart

In [ ]:
total_hb_words    = df['total_hb_words'].sum()
has_a_greek_mapped   = df['has_a_greek_mapped'].sum()
has_a_latvian_mapped = df['has_a_latvian_mapped'].sum()
greek_unmapped   = total_hb_words - has_a_greek_mapped
latvian_unmapped = total_hb_words - has_a_latvian_mapped

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Overall Mapping Coverage (all words across all chapters)', fontsize=14, fontweight='bold')

for ax, (mapped, unmapped, label, color) in zip(axes, [
    (has_a_greek_mapped,   greek_unmapped,   'Greek',   ['#4C72B0', '#AEC6E8']),
    (has_a_latvian_mapped, latvian_unmapped, 'Latvian', ['#DD8452', '#F5C49A']),
]):
    wedges, texts, autotexts = ax.pie(
        [mapped, unmapped],
        labels=[f'Mapped', f'Unmapped'],
        colors=color,
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
    )
    for at in autotexts: at.set_fontsize(11)
    ax.set_title(f'{label} mapping\n({mapped:,} / {total_hb_words:,} words)', fontsize=12)

plt.tight_layout()
plt.savefig(f'{prefix_outputs}overall_coverage_pie.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Per-book usage — Pie charts

In [ ]:
abook_df = df.groupby('book').agg(
    total_hb_words=('total_hb_words', 'sum'),
    total_latvian_words=('total_latvian_words', 'sum'),
    total_greek_words=('total_greek_words', 'sum'),
    leftover_greek=('leftover_greek', 'sum'),
    leftover_latvian=('leftover_latvian', 'sum'),
    chapters=('chapter', 'count'),
).reset_index()
abook_df['greek_pct']   = (abook_df['total_greek_words'] - abook_df['leftover_greek'] ) / abook_df['total_greek_words'] * 100
abook_df['latvian_pct'] = (abook_df['total_latvian_words'] - abook_df['leftover_latvian'] ) / abook_df['total_latvian_words'] * 100

abooks = abook_df['book'].tolist()
n = len(abooks)
ncols = 6
nrows = int(np.ceil(n / ncols))

for lang, col, col_sum, color_mapped, color_un in [
    ('Greek',   'greek_pct', 'leftover_greek',  '#4C72B0', '#AEC6E8'),
    ('Latvian', 'latvian_pct', 'leftover_latvian','#DD8452', '#F5C49A'),
]:
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
    fig.suptitle(f'{lang} Translation words USAGE per Book', fontsize=15, fontweight='bold', y=1.01)
    axes_flat = axes.flatten()

    for i, (_, row) in enumerate(abook_df.iterrows()):
        ax = axes_flat[i]
        pct = row[col]
        ax.pie(
            [pct, 100 - pct],
            colors=[color_mapped, color_un],
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1}
        )
        #ax.set_title(f"{row['book']}\n{pct:.0f}%", fontsize=8)
        #ax.set_title(f"{row['book']}\n{pct:.0f}% \n{int(row[col_sum])}", fontsize=8)
        ax.set_title(f"{row['book']}\n{pct:.0f}%", fontsize=10)
        ax.text(0, -0.3, str(int(row[col_sum])), fontsize=14,
            ha='center', va='top', color=color_un, transform=ax.transData)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'{prefix_outputs}per_book_{lang.lower()}_USE_pies.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Per-book mapping coverage — Pie charts

In [ ]:
book_df = df.groupby('book').agg(
    total_hb_words=('total_hb_words', 'sum'),
    has_a_greek_mapped=('has_a_greek_mapped', 'sum'),
    has_a_latvian_mapped=('has_a_latvian_mapped', 'sum'),
    leftover_greek=('leftover_greek', 'sum'),
    leftover_latvian=('leftover_latvian', 'sum'),
    chapters=('chapter', 'count'),
).reset_index()
book_df['greek_pct']   = book_df['has_a_greek_mapped']   / book_df['total_hb_words'] * 100
book_df['latvian_pct'] = book_df['has_a_latvian_mapped'] / book_df['total_hb_words'] * 100

books = book_df['book'].tolist()
n = len(books)
ncols = 6
nrows = int(np.ceil(n / ncols))

for lang, col, color_mapped, color_un in [
    ('Greek',   'greek_pct',   '#4C72B0', '#AEC6E8'),
    ('Latvian', 'latvian_pct', '#DD8452', '#F5C49A'),
]:
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
    fig.suptitle(f'{lang} Mapping Coverage per Book', fontsize=15, fontweight='bold', y=1.01)
    axes_flat = axes.flatten()

    for i, (_, row) in enumerate(book_df.iterrows()):
        ax = axes_flat[i]
        pct = row[col]
        ax.pie(
            [pct, 100 - pct],
            colors=[color_mapped, color_un],
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1}
        )
        ax.set_title(f"{row['book']}\n{pct:.0f}%", fontsize=8)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'{prefix_outputs}per_book_{lang.lower()}_pies.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Leftover words heatmap — chapters with most leftovers

In [ ]:
# Top chapters by leftover greek
top_greek = df.nlargest(20, 'leftover_greek')[['book','chapter','leftover_greek','leftover_latvian']]
top_latvian = df.nlargest(20, 'leftover_latvian')[['book','chapter','leftover_greek','leftover_latvian']]
top_both = df.nlargest(20, 'leftover_greek')[df.nlargest(20,'leftover_greek')['has_both_leftovers']]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Top chapters by leftover words', fontsize=14, fontweight='bold')

def bar_chart(ax, data, col, title, color):
    labels = data['book'] + ' ch.' + data['chapter'].astype(str)
    ax.barh(labels[::-1], data[col].values[::-1], color=color, edgecolor='white')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Leftover word count')
    ax.tick_params(axis='y', labelsize=8)
    for i, v in enumerate(data[col].values[::-1]):
        ax.text(v + 0.1, i, str(v), va='center', fontsize=7)

bar_chart(axes[0], top_greek,   'leftover_greek',   'Most leftover Greek',   '#4C72B0')
bar_chart(axes[1], top_latvian, 'leftover_latvian', 'Most leftover Latvian', '#DD8452')

# Both leftovers — stacked
labels_both = (top_both['book'] + ' ch.' + top_both['chapter'].astype(str)).values[::-1]
x = np.arange(len(labels_both))
axes[2].barh(x, top_both['leftover_greek'].values[::-1],  color='#4C72B0', label='Greek')
axes[2].barh(x, top_both['leftover_latvian'].values[::-1], color='#DD8452', label='Latvian',
             left=top_both['leftover_greek'].values[::-1])
axes[2].set_yticks(x)
axes[2].set_yticklabels(labels_both, fontsize=8)
axes[2].set_title('Chapters with BOTH leftovers (stacked)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Leftover word count')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{prefix_outputs}top_leftover_chapters.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Chapter-level heatmap — leftover density per book

In [ ]:
# Pivot: rows=books, cols=chapters
for lang, col, cmap_name, title in [
    ('Greek',   'leftover_greek',   'Blues',   'Leftover Greek words per chapter'),
    ('Latvian', 'leftover_latvian', 'Oranges', 'Leftover Latvian words per chapter'),
]:
    pivot = df.pivot_table(index='book', columns='chapter', values=col, fill_value=0)
    pivot = pivot.reindex(sorted(pivot.index))  # alphabetical

    fig, ax = plt.subplots(figsize=(max(14, pivot.shape[1] * 0.35), max(6, pivot.shape[0] * 0.35)))
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap_name, interpolation='nearest')
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, fontsize=6, rotation=90)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel('Chapter')
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.6, label='Leftover word count')
    plt.tight_layout()
    plt.savefig(f'{prefix_outputs}heatmap_{lang.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Scatter — chapters with BOTH leftovers

In [ ]:
both = df[df['has_both_leftovers']].copy()

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    both['leftover_greek'],
    both['leftover_latvian'],
    c=both['total_hb_words'],
    cmap='viridis',
    alpha=0.75,
    edgecolors='white',
    linewidth=0.5,
    s=80
)
plt.colorbar(scatter, label='Total Hebrew words in chapter')

# Annotate notable outliers (top 10 by sum of leftovers)
both['total_leftover'] = both['leftover_greek'] + both['leftover_latvian']
top = both.nlargest(10, 'total_leftover')
for _, row in top.iterrows():
    ax.annotate(
        f"{row['book']} {row['chapter']}",
        (row['leftover_greek'], row['leftover_latvian']),
        fontsize=7, alpha=0.85,
        xytext=(4, 4), textcoords='offset points'
    )

ax.set_xlabel('Leftover Greek words', fontsize=11)
ax.set_ylabel('Leftover Latvian words', fontsize=11)
ax.set_title('Chapters with BOTH leftover Greek & Latvian words', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{prefix_outputs}scatter_both_leftovers.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total chapters with both leftovers: {len(both)}")

## 6. Summary table — books with most leftover issues

In [ ]:
summary = book_df.sort_values('leftover_greek', ascending=False)
summary['greek_pct']   = summary['greek_pct'].round(1)
summary['latvian_pct'] = summary['latvian_pct'].round(1)
summary = summary[['book','chapters','total_hb_words','greek_pct','latvian_pct','leftover_greek','leftover_latvian']]
summary.columns = ['Book','Chapters','Words','Greek mapped %','Latvian mapped %','Leftover Greek','Leftover Latvian']
#book_df.to_csv(f'bbl_mapings_{datetime.datetime.now}.csv')
book_df.to_csv(f'{prefix_outputs}bbl_mapings_coverage.csv')
print(summary.to_string(index=False))